# Equazioni differenziali con SymPy e SciPy

Questo notebook ti aiuta a **scrivere**, **risolvere analiticamente** e **integrare numericamente** equazioni differenziali ordinarie (ODE).

| Sezione | Cosa fa |
|---------|---------|
| 1. Setup | Import e helper |
| 2. Scrivere un'ODE | Sintassi SymPy |
| 3. Primo ordine | Separabili, lineari |
| 4. Secondo ordine | Omogenee e non |
| 5. Problemi di Cauchy | Condizioni iniziali |
| 6. Sistemi | Più equazioni |
| 7. Numerico | `solve_ivp` + grafici |

## 1. Setup

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

sp.init_printing(use_unicode=True)

def mostra(eq, titolo=None):
    """Stampa un'equazione o una soluzione in modo leggibile."""
    if titolo:
        print(titolo)
    display(eq)

print("SymPy", sp.__version__, "| NumPy", np.__version__)

## 2. Come scrivere un'equazione differenziale

Passi tipici con SymPy:

1. Definisci la variabile indipendente `t` (o `x`)
2. Definisci la funzione sconosciuta `y = Function('y')`
3. Scrivi l'ODE come `Eq(lhs, rhs)` usando `y(t).diff(t)` per le derivate
4. Risolvi con `dsolve`

In [ ]:
t = sp.symbols('t', real=True)
y = sp.Function('y')

# y' = -2y  →  dy/dt + 2y = 0
ode = sp.Eq(y(t).diff(t), -2 * y(t))
mostra(ode, "Equazione:")

sol = sp.dsolve(ode, y(t))
mostra(sol, "Soluzione generale:")

### Template riusabile

Copia questa cella e modifica `lhs` / `rhs` (o l'intera `Eq`).

In [ ]:
def risolvi_ode(equazione, funzione=None, ics=None):
    """
    Risolve un'ODE (o un sistema) con SymPy.

    Parametri
    ---------
    equazione : Eq o lista di Eq
    funzione  : Function o lista (opzionale; SymPy spesso la deduce)
    ics       : dict condizioni iniziali, es. {y(0): 1, y(t).diff(t).subs(t, 0): 0}
    """
    kwargs = {}
    if funzione is not None:
        kwargs['func'] = funzione
    if ics is not None:
        kwargs['ics'] = ics
    return sp.dsolve(equazione, **kwargs)


# Esempio: y' + y = e^{-t}
ode_esempio = sp.Eq(y(t).diff(t) + y(t), sp.exp(-t))
mostra(risolvi_ode(ode_esempio, y(t)), "Soluzione:")

## 3. Equazioni del primo ordine

### 3.1 Separabile — $y' = ky$

In [ ]:
k = sp.symbols('k', real=True)
ode_sep = sp.Eq(y(t).diff(t), k * y(t))
mostra(ode_sep, "ODE:")
mostra(sp.dsolve(ode_sep, y(t)), "Soluzione:")

### 3.2 Lineare — $y' + P(t)y = Q(t)$

Esempio: $y' + 2ty = t$

In [ ]:
ode_lin = sp.Eq(y(t).diff(t) + 2 * t * y(t), t)
mostra(ode_lin, "ODE:")
sol_lin = sp.dsolve(ode_lin, y(t))
mostra(sol_lin, "Soluzione:")
mostra(sp.simplify(sol_lin.rhs), "rhs semplificato:")

### 3.3 Verifica della soluzione

`checkodesol` restituisce `(True/False, residuo)`.

In [ ]:
ok, residuo = sp.checkodesol(ode_lin, sol_lin)
print("Soluzione valida:", ok)
mostra(sp.simplify(residuo), "Residuo:")

## 4. Equazioni del secondo ordine

### 4.1 Omogenea a coefficienti costanti — $y'' + y = 0$

In [ ]:
ode2 = sp.Eq(y(t).diff(t, 2) + y(t), 0)
mostra(ode2, "ODE:")
mostra(sp.dsolve(ode2, y(t)), "Soluzione:")

### 4.2 Non omogenea — $y'' + y = \sin(2t)$

In [ ]:
ode2_nh = sp.Eq(y(t).diff(t, 2) + y(t), sp.sin(2 * t))
mostra(ode2_nh, "ODE:")
mostra(sp.dsolve(ode2_nh, y(t)), "Soluzione:")

### 4.3 Smorzamento — $y'' + 2\zeta\omega y' + \omega^2 y = 0$

In [ ]:
omega, zeta = sp.symbols('omega zeta', positive=True)
ode_smorz = sp.Eq(
    y(t).diff(t, 2) + 2 * zeta * omega * y(t).diff(t) + omega**2 * y(t),
    0,
)
mostra(ode_smorz, "ODE:")
mostra(sp.dsolve(ode_smorz, y(t)), "Soluzione generale:")

## 5. Problema di Cauchy (condizioni iniziali)

Passa `ics={...}` a `dsolve` per fissare le costanti.

In [ ]:
# y' = -2y,  y(0) = 3
ode_ivp = sp.Eq(y(t).diff(t), -2 * y(t))
sol_ivp = sp.dsolve(ode_ivp, y(t), ics={y(0): 3})
mostra(sol_ivp, "Soluzione particolare:")

In [ ]:
# y'' + y = 0,  y(0)=1,  y'(0)=0  →  cos(t)
ode_ivp2 = sp.Eq(y(t).diff(t, 2) + y(t), 0)
sol_ivp2 = sp.dsolve(
    ode_ivp2,
    y(t),
    ics={y(0): 1, y(t).diff(t).subs(t, 0): 0},
)
mostra(sol_ivp2, "Soluzione:")

### Da simbolico a funzione numerica

Usa `lambdify` per valutare e plottare la soluzione analitica.

In [ ]:
y_num = sp.lambdify(t, sol_ivp2.rhs, modules='numpy')
tt = np.linspace(0, 4 * np.pi, 400)

plt.figure(figsize=(8, 4))
plt.plot(tt, y_num(tt), color='#1a5f7a', lw=2)
plt.xlabel('t')
plt.ylabel('y(t)')
plt.title(r"$y'' + y = 0$,  $y(0)=1$, $y'(0)=0$")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Sistemi di equazioni

Esempio: oscillatore armonico come sistema del primo ordine

$$
\begin{cases}
x' = v \\
v' = -\omega^2 x
\end{cases}
$$

In [ ]:
x, v = sp.Function('x'), sp.Function('v')
omega0 = sp.symbols('omega', positive=True)

sistema = [
    sp.Eq(x(t).diff(t), v(t)),
    sp.Eq(v(t).diff(t), -omega0**2 * x(t)),
]

sol_sys = sp.dsolve(
    sistema,
    [x(t), v(t)],
    ics={x(0): 1, v(0): 0},
)
for s in sol_sys:
    display(s)

## 7. Soluzione numerica con SciPy

Quando SymPy non trova una forma chiusa (o l'ODE è non lineare / stiff), usa `solve_ivp`.

In [ ]:
def integra_e_plot(
    f,
    t_span,
    y0,
    *,
    t_eval=None,
    labels=None,
    titolo=None,
    metodo='RK45',
):
    """
    Integra y' = f(t, y) e disegna le componenti.

    f      : callable (t, y) -> dy/dt
    t_span : (t0, tf)
    y0     : condizione iniziale (scalare o array)
    """
    if t_eval is None:
        t_eval = np.linspace(t_span[0], t_span[1], 500)

    sol = solve_ivp(f, t_span, np.atleast_1d(y0).astype(float),
                    t_eval=t_eval, method=metodo, dense_output=False)

    if not sol.success:
        raise RuntimeError(sol.message)

    n = sol.y.shape[0]
    if labels is None:
        labels = [f'y[{i}]' for i in range(n)]

    plt.figure(figsize=(8, 4))
    for i in range(n):
        plt.plot(sol.t, sol.y[i], lw=2, label=labels[i])
    plt.xlabel('t')
    plt.ylabel('stato')
    if titolo:
        plt.title(titolo)
    if n > 1:
        plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return sol

### 7.1 ODE scalare non lineare — $y' = y(1 - y)$ (logistica)

In [ ]:
def logistica(t, y):
    return y * (1 - y)

integra_e_plot(
    logistica,
    (0, 8),
    y0=0.1,
    labels=['y'],
    titolo=r"$y' = y(1-y)$,  $y(0)=0.1$",
)

### 7.2 Confronta analitico vs numerico

In [ ]:
# Analitico: y' = -2y, y(0)=3  →  y = 3 e^{-2t}
sol_a = sp.dsolve(sp.Eq(y(t).diff(t), -2 * y(t)), y(t), ics={y(0): 3})
f_a = sp.lambdify(t, sol_a.rhs, 'numpy')

def f_num(t, y):
    return -2 * y

sol_n = solve_ivp(f_num, (0, 3), [3.0], t_eval=np.linspace(0, 3, 200))

plt.figure(figsize=(8, 4))
plt.plot(sol_n.t, f_a(sol_n.t), 'k-', lw=2.5, label='analitico')
plt.plot(sol_n.t, sol_n.y[0], '--', color='#c45c26', lw=2, label='numerico')
plt.xlabel('t')
plt.ylabel('y')
plt.title(r"Confronto: $y'=-2y$, $y(0)=3$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

err = np.max(np.abs(f_a(sol_n.t) - sol_n.y[0]))
print(f"Errore massimo |analitico - numerico|: {err:.2e}")

### 7.3 Sistema — Lotka–Volterra (preda–predatore)

$$
\begin{cases}
x' = ax - bxy \\
y' = -cy + dxy
\end{cases}
$$

In [ ]:
a, b, c, d = 1.1, 0.4, 0.4, 0.1

def lotka_volterra(t, z):
    x, y = z
    return [a * x - b * x * y, -c * y + d * x * y]

sol_lv = integra_e_plot(
    lotka_volterra,
    (0, 40),
    y0=[10, 5],
    labels=['prede x', 'predatori y'],
    titolo='Lotka–Volterra',
)

# Piano delle fasi
plt.figure(figsize=(5, 5))
plt.plot(sol_lv.y[0], sol_lv.y[1], color='#1a5f7a', lw=1.5)
plt.xlabel('prede')
plt.ylabel('predatori')
plt.title('Piano delle fasi')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Laboratorio — scrivi la tua equazione

Modifica le celle sotto: cambia ODE, condizioni iniziali e intervallo.

In [ ]:
# --- SIMBOLICO ---
t = sp.symbols('t')
y = sp.Function('y')

# Scrivi qui la tua ODE, es. y'' - 3y' + 2y = 0
mia_ode = sp.Eq(y(t).diff(t, 2) - 3 * y(t).diff(t) + 2 * y(t), 0)

# Condizioni iniziali (None = soluzione generale)
mie_ics = {y(0): 1, y(t).diff(t).subs(t, 0): 0}

mia_sol = sp.dsolve(mia_ode, y(t), ics=mie_ics)
mostra(mia_ode, "La tua ODE:")
mostra(mia_sol, "Soluzione:")

ok, _ = sp.checkodesol(mia_ode, mia_sol)
print("Verifica:", "OK" if ok else "ATTENZIONE")

In [ ]:
# --- NUMERICO ---
# Forma: y' = f(t, y)   oppure   [y', v'] = f(t, [y, v]) per il 2° ordine

def mia_f(t, z):
    # Esempio: y'' - 3y' + 2y = 0  →  z = [y, y']
    y, yp = z
    return [yp, 3 * yp - 2 * y]

integra_e_plot(
    mia_f,
    t_span=(0, 5),
    y0=[1.0, 0.0],
    labels=['y', "y'"],
    titolo='La tua ODE (numerico)',
)

## Promemoria sintassi

| Concetto | SymPy |
|----------|-------|
| Derivata prima | `y(t).diff(t)` o `sp.diff(y(t), t)` |
| Derivata n-esima | `y(t).diff(t, n)` |
| Equazione | `sp.Eq(lhs, rhs)` |
| Risolvi | `sp.dsolve(ode, y(t))` |
| Con CI | `sp.dsolve(ode, y(t), ics={y(0): ...})` |
| Verifica | `sp.checkodesol(ode, sol)` |
| Numerico | `scipy.integrate.solve_ivp(f, t_span, y0)` |

**Dipendenze:** `pip install sympy numpy scipy matplotlib`